# FD-24 Сессия 2: Факториальное сравнение бэкбонов и контекста (Слоты 5–7)
- **Данные:** полная смесь (`--train_mix full`)
- **Слоты:**
  - Слот 5: ModernBERT-base @1024 (контроль домена: кодовый vs общий энкодер)
  - Слот 6: GTE-ModernBERT-base @2048 (`--gradient_checkpointing`, парный претрейн + снятие усечения)
  - Слот 7: ModernBERT-base @2048 (`--gradient_checkpointing`, контроль контекста; пропуск при паритете 5 и 6)
- **Предполётка:** Smoke test, overfit-test 200 строк и пилот 100 шагов FP16 с автоматической проверкой NaN.

In [ ]:
%cd /kaggle/working
import os
from kaggle_secrets import UserSecretsClient
try:
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _repo = f"https://{_tok}@github.com/o1uh/FreshDoc.git"
    print("clone: private (token)")
except Exception:
    _repo = "https://github.com/o1uh/FreshDoc.git"
    print("clone: public fallback (repo must be public)")

!if [ ! -d FreshDoc ]; then git clone --quiet {_repo} > /tmp/clone.log 2>&1 || (echo "CLONE FAILED:"; sed -E 's#(https?://)[^@]+@#\1***@#g' /tmp/clone.log); fi
if '_tok' in locals(): del _tok
if '_repo' in locals(): del _repo

%cd /kaggle/working/FreshDoc
!git checkout feat/FD-24-kaggle-experiments 2>/dev/null || !git checkout main 2>/dev/null || true
!git pull --ff-only || echo "PULL SKIPPED: убедись, что код свежий"

import os
os.environ["NCCL_P2P_DISABLE"] = "1"
!python -c "import transformers, sys; from packaging import version; sys.exit(0 if version.parse(transformers.__version__) >= version.parse('4.48.0') else 1)" || pip install --quiet --no-deps "transformers>=4.48.0"
!nvidia-smi

In [ ]:
import os, glob
import pandas as pd
_cands = glob.glob("/kaggle/input/**/dataset_v1.parquet", recursive=True)
assert _cands, "dataset_v1.parquet не найден в /kaggle/input (добавь датасет в Inputs блокнота)"
_p = _cands[0]
os.environ["DATA_PATH"] = _p
print(f"Dataset path: {_p}")

_df = pd.read_parquet(_p, columns=["split"])
assert len(_df) == 27203, f"dataset rows {len(_df)} != 27203"
_exp = {"train": 21997, "val": 559, "test_real": 2895, "test_synth": 1552, "gold": 200}
_got = _df["split"].value_counts().to_dict()
assert _got == _exp, f"split mismatch: {_got}"
print("dataset OK:", _got)
del _df

In [ ]:
# Копирование runs.csv из Session 1 (inputs) в working при раздельном запуске сессий
import os, glob, shutil
local_runs = os.path.abspath('reports/runs.csv')
input_runs = sorted(glob.glob('/kaggle/input/**/runs.csv', recursive=True), key=os.path.getmtime, reverse=True)
if input_runs:
    print(f'Импорт истории runs.csv из Session 1: {input_runs[0]} -> {local_runs}')
    shutil.copy(input_runs[0], local_runs)
else:
    print('runs.csv из предыдущей сессии не найден, используется базовый runs.csv репозитория')


In [ ]:
# Генерация бейзлайнов M0 и M1 (подготовка для evaluate и стэкинга)
!python scripts/baselines_m0_m1.py --dataset "$DATA_PATH" \
  --out_m0 /kaggle/working/m0.csv --out_m1 /kaggle/working/m1.csv \
  --out_model /kaggle/working/m1_hgb.joblib --out_report /kaggle/working/m1_report.json

### Предполётная проверка ModernBERT

In [ ]:
# Предполётка: smoke-тест весов (granite 48M, UniXcoder, ModernBERT, GTE)
import gc, torch
from transformers import AutoModel, AutoTokenizer
for m in ["ibm-granite/granite-embedding-small-english-r2",
          "microsoft/unixcoder-base",
          "answerdotai/ModernBERT-base",
          "Alibaba-NLP/gte-modernbert-base"]:
    tok = AutoTokenizer.from_pretrained(m)
    mod = AutoModel.from_pretrained(m).to("cuda")
    out = mod(**tok("def test(): pass", return_tensors="pt").to("cuda"))
    print(f"Smoke OK: {m:45s} shape={tuple(out.last_hidden_state.shape)}")
    del mod, out, tok
    gc.collect()
    torch.cuda.empty_cache()
print("GPU freed.")


In [ ]:
# Предполётка: тест на переобучение ModernBERT (200 строк x 20 эпох)
# Ожидание: PR-AUC >= 0.99 (критерий R10)
!python scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name answerdotai/ModernBERT-base \
  --output_dir /kaggle/working/checkpoints/pilot_modernbert_overfit --overfit_test
!rm -rf /kaggle/working/checkpoints/pilot_modernbert_overfit

In [ ]:
# Предполётка @1024: пилот 100 шагов FP16 на ModernBERT с проверкой NaN и замером с/шаг
# Если уходит в NaN — стоп-критерий: снизить lr до 1e-5 (--lr 1e-5)
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name answerdotai/ModernBERT-base \
  --max_len 1024 --pilot_steps 100 --output_dir /kaggle/working/checkpoints/pilot_modernbert_100
!rm -rf /kaggle/working/checkpoints/pilot_modernbert_100

# Предполётка @2048: ранняя валидация аллокации памяти и исключение риска CUDA OOM
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name Alibaba-NLP/gte-modernbert-base \
  --max_len 2048 --gradient_checkpointing --pilot_steps 20 --output_dir /kaggle/working/checkpoints/pilot_gte_2048
!rm -rf /kaggle/working/checkpoints/pilot_gte_2048


### Слоты 5–7

In [ ]:
# ==============================================================================
# Слот 5: ModernBERT-base @1024 (контроль домена)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name answerdotai/ModernBERT-base \
  --max_len 1024 --train_mix full \
  --output_dir /kaggle/working/checkpoints/slot5_modernbert1024 --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot5_modernbert1024/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot5_modernbert1024/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot5_modernbert1024/best \
  --dataset "$DATA_PATH"  --out /kaggle/working/checkpoints/slot5_modernbert1024/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot5_modernbert1024/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot5_modernbert1024/metrics.json --out_table /kaggle/working/checkpoints/slot5_modernbert1024/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot5_modernbert1024/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot5_modernbert1024/table_e1.csv

In [ ]:
# ==============================================================================
# Слот 6: GTE-ModernBERT-base @2048 (снятие усечения + relevance претрейн)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name Alibaba-NLP/gte-modernbert-base \
  --max_len 2048 --gradient_checkpointing --train_mix full \
  --output_dir /kaggle/working/checkpoints/slot6_gte2048 --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot6_gte2048/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot6_gte2048/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot6_gte2048/best \
  --dataset "$DATA_PATH" --batch_size 16 --out /kaggle/working/checkpoints/slot6_gte2048/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot6_gte2048/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot6_gte2048/metrics.json --out_table /kaggle/working/checkpoints/slot6_gte2048/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot6_gte2048/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot6_gte2048/table_e1.csv

In [ ]:
# ==============================================================================
# Слот 7: ModernBERT-base @2048 (контроль контекста)
# Опциональный слот: если PR-AUC слота 6 не превосходит слот 5 на >= 0.005, ячейку можно пропустить для экономии ~50 мин GPU
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name answerdotai/ModernBERT-base \
  --max_len 2048 --gradient_checkpointing --train_mix full \
  --output_dir /kaggle/working/checkpoints/slot7_modernbert2048 --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot7_modernbert2048/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot7_modernbert2048/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot7_modernbert2048/best \
  --dataset "$DATA_PATH" --batch_size 16 --out /kaggle/working/checkpoints/slot7_modernbert2048/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot7_modernbert2048/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot7_modernbert2048/metrics.json --out_table /kaggle/working/checkpoints/slot7_modernbert2048/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot7_modernbert2048/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot7_modernbert2048/table_e1.csv

### Сводка Сессии 2

In [ ]:
import pandas as pd
df_runs = pd.read_csv("reports/runs.csv")
print(df_runs.to_string(index=False))